# Fase 3 · Representación VSM de frecuencia (scikit-learn)

Este notebook construye las representaciones de **frecuencia** (conteo de términos, *term frequency* cruda) del corpus normalizado (`data/scientific_normalized_corpus.tsv`) con `CountVectorizer` de scikit-learn.

**Objetivo:** generar las 4 configuraciones de frecuencia del plan, {título, abstract} × {unigrama, bigrama}, y guardar cada vectorizador con su matriz en `models/{campo}_{ngrama}_frequency.pkl`, con la misma estructura que los `.pkl` de TF-IDF (notebook 03) y binarios (notebook 04).

## ¿Qué es la representación de frecuencia?

Cada documento $d_i$ es un vector con una posición por término $t_j$ del vocabulario, y el valor es **cuántas veces aparece** el término en el documento:

$$
x_{ij} = \mathrm{tf}(t_j, d_i) = \text{número de apariciones de } t_j \text{ en } d_i
$$

Comparada con las otras dos representaciones del plan:

| Representación | Valor de $x_{ij}$ | Qué captura |
|---|---|---|
| **Frecuencia** | **número de veces que $t_j$ aparece en $d_i$** | **presencia + repetición** |
| Binaria | 1 si aparece, 0 si no | solo presencia |
| TF-IDF | frecuencia × idf (rareza del término en el corpus) | presencia + repetición + poder discriminante |

Las tres están relacionadas: la binaria es `frecuencia > 0`, y TF-IDF (antes de normalizar) es `frecuencia × idf`. Ambas relaciones se verifican en la sección 7.

## Decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Clase | `CountVectorizer()` (con `binary=False`, por defecto) | Lo pide el plan (sección Fase 3) |
| Texto de entrada | `title_norm` y `abstract_norm` | Igual que TF-IDF y binaria: se ajusta sobre el texto ya normalizado |
| `lowercase` | `False` | La normalización ya pasó todo a minúsculas |
| `token_pattern` | `r"\S+"` | Los tokens ya vienen separados por espacios y no se deben perder tokens de un carácter |
| `stop_words` | `None` | Ya se quitaron en la normalización |
| `ngram_range` | `(1, 1)` unigramas, `(2, 2)` bigramas | Lo pide el plan; bigramas formados tras quitar stopwords |
| `min_df` / `max_df` | Por defecto (`1` y `1.0`) | Mismo vocabulario que TF-IDF y binaria, así las 3 representaciones son comparables término a término |
| Normalización | **No** se guarda normalizada | La matriz guardada son los conteos crudos. En la Fase 4 la similitud coseno debe dividir entre las normas (ver sección 8) |
| Formato guardado | `pickle` con vectorizador, matriz CSR y `doc_ids` | Mismas llaves que los otros `.pkl`, para que `retrieve()` cargue cualquier tipo igual |

## 1. Configuración

Se importan las librerías y se definen las rutas. `FIELDS` y `NGRAMS` son los mismos diccionarios de los notebooks 03 y 04; solo cambia `REPR = "frequency"`, que va en el nombre de cada archivo.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NORM_PATH = ROOT / "data" / "scientific_normalized_corpus.tsv"
MODELS_DIR = ROOT / "models"
REPORT_PATH = ROOT / "results" / "frequency_stats.md"
MODELS_DIR.mkdir(exist_ok=True)

# Campo del CLI -> columna normalizada del TSV
FIELDS = {"title": "title_norm", "abstract": "abstract_norm"}
# Tipo de n-grama del CLI -> ngram_range de scikit-learn
NGRAMS = {"unigram": (1, 1), "bigram": (2, 2)}
REPR = "frequency"

print("scikit-learn", sklearn.__version__)

scikit-learn 1.9.1


## 2. Carga del corpus normalizado

Se lee el TSV como texto (`dtype=str`) y con `keep_default_na=False` para que un campo vacío quede como `""` y no como `NaN` (el vectorizador fallaría con `NaN`). La lista `doc_ids` guarda el `arxiv_id` de cada fila: la fila *i* de cada matriz corresponde a `doc_ids[i]`.

In [2]:
corpus = pd.read_csv(NORM_PATH, sep="\t", dtype=str, keep_default_na=False)
assert corpus["arxiv_id"].is_unique
doc_ids = corpus["arxiv_id"].tolist()

for col in FIELDS.values():
    empty = (corpus[col].str.strip() == "").sum()
    print(f"{col}: {len(corpus)} documentos, {empty} vacíos")

corpus[["arxiv_id", "title_norm"]].head(3)

title_norm: 500 documentos, 0 vacíos
abstract_norm: 500 documentos, 0 vacíos


,arxiv_id,title_norm
0,2610.00840,contextual trajectory incremental contextual d...
1,2610.00833,veritygate four gate schema level faithfulness...
2,2610.00827,verbalize internal probabilities couple large ...


## 3. Ejemplo pequeño: unigramas y bigramas por frecuencia

Antes de usar el corpus real, se ve qué produce `CountVectorizer` con 3 documentos de juguete (ya "normalizados"), primero con unigramas y luego con bigramas.

In [3]:
toy = [
    "model learn model representation",
    "text model text text classification",
    "image classification",
]
docs = ["doc0", "doc1", "doc2"]
params = dict(lowercase=False, token_pattern=r"\S+")

uni_vec = CountVectorizer(**params, ngram_range=(1, 1)).fit(toy)
X_uni = uni_vec.transform(toy)
toy_uni = pd.DataFrame(X_uni.toarray(), index=docs, columns=uni_vec.get_feature_names_out())
toy_uni["total"] = toy_uni.sum(axis=1)
toy_uni

,classification,image,learn,model,representation,text,total
doc0,0,0,1,2,1,0,4
doc1,1,0,0,1,0,3,5
doc2,1,1,0,0,0,0,2


Qué se observa con unigramas:

- **Vocabulario:** los 6 términos distintos de los 3 documentos, en orden alfabético; cada uno es una columna.
- **`doc0`:** `model` aparece 2 veces → vale 2. En la binaria valdría 1.
- **`doc1`:** `text` aparece 3 veces → vale 3, así que domina el vector de ese documento.
- **`total`:** la suma de la fila es el número de tokens del documento (4, 5 y 2). Nada se pierde: cada token suma 1 en su columna.

In [4]:
bi_vec = CountVectorizer(**params, ngram_range=(2, 2)).fit(toy)
toy_bi = pd.DataFrame(bi_vec.transform(toy).toarray(), index=docs, columns=bi_vec.get_feature_names_out())
toy_bi["total"] = toy_bi.sum(axis=1)
toy_bi

,image classification,learn model,model learn,model representation,model text,text classification,text model,text text,total
doc0,0,1,1,1,0,0,0,0,3
doc1,0,0,0,0,1,1,1,1,4
doc2,1,0,0,0,0,0,0,0,1


Con bigramas cada columna es un par de tokens consecutivos:

- `doc1` = `text model text text classification` produce los bigramas `text model`, `model text`, `text text`, `text classification`, cada uno una vez.
- Un documento con $n$ tokens produce $n - 1$ bigramas, por eso `total` es 3, 4 y 1.
- Los bigramas repetidos son mucho más raros que los unigramas repetidos, así que en bigramas la frecuencia se parece mucho a la binaria (se mide en la sección 6).

## 4. Vectorizador de frecuencia

`build_vectorizer` es la única fábrica de vectorizadores, para que las 4 configuraciones usen exactamente los mismos parámetros salvo `ngram_range`. Son los mismos parámetros de tokenización que en los notebooks 03 y 04, así que el vocabulario debe coincidir (se verifica en la sección 7).

In [5]:
def build_vectorizer(ngram_range) -> CountVectorizer:
    return CountVectorizer(
        lowercase=False,       # el texto ya está en minúsculas
        token_pattern=r"\S+",  # los tokens ya vienen separados por espacios
        ngram_range=ngram_range,
        binary=False,          # se guarda el conteo de apariciones
    )


def model_path(field: str, ngram: str, repr_: str = REPR) -> Path:
    return MODELS_DIR / f"{field}_{ngram}_{repr_}.pkl"

## 5. Paso a paso con una configuración (abstract · unigrama)

Antes del ciclo de las 4 configuraciones, se desglosa lo que hace `fit_transform` sobre una sola.

### 5.1 `fit`: construir el vocabulario

`fit` recorre todos los abstracts, los parte en tokens con `token_pattern` y asigna a cada término distinto un índice de columna. El resultado queda en `vocabulary_` (término → columna).

In [6]:
demo_vec = build_vectorizer(NGRAMS["unigram"])
demo_vec.fit(corpus["abstract_norm"])

demo_terms = demo_vec.get_feature_names_out()
print("Tamaño del vocabulario:", len(demo_terms))
print("Columna de 'transformer':", demo_vec.vocabulary_.get("transformer"))

Tamaño del vocabulario: 7958
Columna de 'transformer': 7340


### 5.2 `transform`: construir la matriz documento-término

`transform` cuenta cuántas veces aparece cada término en cada documento. La salida es una matriz **dispersa** (`scipy.sparse`): solo se guardan las celdas distintas de cero. Se convierte a CSR (`tocsr()`) para que extraer filas y multiplicar sea rápido.

`nnz` (celdas distintas de cero) es el número de pares (documento, término distinto); la **suma** de la matriz es el número total de tokens del corpus. La diferencia entre ambos mide cuánta repetición hay.

In [7]:
X_demo = demo_vec.transform(corpus["abstract_norm"]).tocsr()

print("Forma (documentos, términos):", X_demo.shape)
print("Celdas distintas de cero (nnz):", X_demo.nnz)
print("Suma de la matriz (tokens totales):", X_demo.sum())
print("Valor máximo de una celda:", X_demo.max())
print("Tipo de dato:", X_demo.dtype)

Forma (documentos, términos): (500, 7958)
Celdas distintas de cero (nnz): 52894
Suma de la matriz (tokens totales): 73532
Valor máximo de una celda: 28
Tipo de dato: int64


### 5.3 Distribución de los conteos

Cuántas celdas no nulas valen 1, 2, 3… La mayoría de los términos aparece una sola vez en su documento; esas celdas son idénticas en la representación binaria. Las diferencias entre frecuencia y binaria vienen del resto.

In [8]:
values, freqs = np.unique(X_demo.data, return_counts=True)
dist = pd.DataFrame({"conteo": values, "celdas": freqs})
dist["%_celdas"] = (100 * dist["celdas"] / X_demo.nnz).round(2)
dist.head(10)

,conteo,celdas,%_celdas
0,1,41261,78.01
1,2,7238,13.68
2,3,2297,4.34
3,4,1020,1.93
4,5,496,0.94
5,6,260,0.49
6,7,127,0.24
7,8,70,0.13
8,9,47,0.09
9,10,27,0.05


### 5.4 Leer una fila

La fila 0 es el vector del primer documento. En formato CSR, `indices` son las columnas no nulas y `data` sus conteos. Se comprueba que la suma de la fila es el número de tokens de `abstract_norm`, y se listan los términos más repetidos.

In [9]:
def top_terms(model_vec, matrix, row: int, k: int = 10) -> pd.DataFrame:
    vec = matrix[row]
    terms = model_vec.get_feature_names_out()
    order = np.lexsort((terms[vec.indices], -vec.data))[:k]  # conteo desc, término asc
    return pd.DataFrame({"termino": terms[vec.indices[order]], "frecuencia": vec.data[order]})


row = X_demo[0]
print(corpus.loc[0, "arxiv_id"], "|", corpus.loc[0, "title"])
print("Tokens en abstract_norm:", len(corpus.loc[0, "abstract_norm"].split()))
print("Suma de la fila:", row.sum())
print("Términos distintos (nnz de la fila):", row.nnz)
top_terms(demo_vec, X_demo, 0)

2610.00840 | Contextual trajectory and incremental contextual displacement: Towards using LLMs to understand dynamic, utterance-specific meaning construction
Tokens en abstract_norm: 141
Suma de la fila: 141
Términos distintos (nnz de la fila): 103


,termino,frecuencia
0,token,6
1,sentence,5
2,trajectory,5
3,embedding,3
4,garden,3
5,information,3
6,path,3
7,use,3
8,utterance,3
9,wise,3


## 6. Ajuste y guardado de las 4 configuraciones

Para cada combinación campo × n-grama:

1. Se crea un vectorizador nuevo con `build_vectorizer`.
2. `fit_transform` aprende el vocabulario y genera la matriz de conteos en un solo paso.
3. Se guarda un diccionario con el vectorizador (necesario para transformar consultas en la Fase 4), la matriz y `doc_ids`.
4. Se calculan las estadísticas que pide el plan (tamaño del vocabulario y dispersión) y dos columnas propias de la frecuencia:
   - `tokens_por_doc`: suma de la matriz entre el número de documentos.
   - `celdas_>1_%`: porcentaje de celdas no nulas con conteo mayor que 1, es decir, donde la frecuencia y la binaria difieren.

In [10]:
rows = []
for field, col in FIELDS.items():
    for ngram, ngram_range in NGRAMS.items():
        vectorizer = build_vectorizer(ngram_range)
        matrix = vectorizer.fit_transform(corpus[col]).tocsr()

        payload = {
            "field": field,
            "ngram": ngram,
            "repr": REPR,
            "column": col,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "doc_ids": doc_ids,
        }
        path = model_path(field, ngram)
        with open(path, "wb") as f:
            pickle.dump(payload, f)

        n_docs, n_terms = matrix.shape
        empty_rows = int((matrix.getnnz(axis=1) == 0).sum())
        rows.append({
            "config": path.stem,
            "documentos": n_docs,
            "vocabulario": n_terms,
            "no_ceros": matrix.nnz,
            "densidad_%": round(100 * matrix.nnz / (n_docs * n_terms), 3),
            "dispersion_%": round(100 * (1 - matrix.nnz / (n_docs * n_terms)), 3),
            "terminos_por_doc": round(matrix.nnz / n_docs, 1),
            "tokens_por_doc": round(matrix.sum() / n_docs, 1),
            "conteo_max": int(matrix.max()),
            "celdas_>1_%": round(100 * (matrix.data > 1).sum() / matrix.nnz, 2),
            "docs_sin_terminos": empty_rows,
        })
        print("Guardado", path.relative_to(ROOT))

stats = pd.DataFrame(rows).set_index("config")
stats

Guardado models/title_unigram_frequency.pkl
Guardado models/title_bigram_frequency.pkl


Guardado models/abstract_unigram_frequency.pkl


Guardado models/abstract_bigram_frequency.pkl


,documentos,vocabulario,no_ceros,densidad_%,dispersion_%,terminos_por_doc,tokens_por_doc,conteo_max,celdas_>1_%,docs_sin_terminos
config,,,,,,,,,,
title_unigram_frequency,500,1863,4262,0.458,99.542,8.5,8.7,3,1.64,0
title_bigram_frequency,500,3545,3830,0.216,99.784,7.7,7.7,2,0.16,0
abstract_unigram_frequency,500,7958,52894,1.329,98.671,105.8,147.1,28,21.99,0
abstract_bigram_frequency,500,60073,69949,0.233,99.767,139.9,146.1,12,3.48,0


## 7. Verificaciones

Para cada `.pkl` guardado:

1. Se recarga y tiene la forma y los `doc_ids` correctos.
2. **Los conteos cuadran con el texto:** la suma de cada fila es el número de tokens del documento (unigramas) o el número de tokens − 1 (bigramas).
3. **Relación con la binaria:** `frecuencia > 0` es igual a la matriz del `.pkl` binario.
4. **Relación con TF-IDF:** mismo vocabulario, y `normalize(frecuencia × idf)` reproduce la matriz TF-IDF guardada. Confirma que TF-IDF es la frecuencia ponderada por idf.
5. Transformar de nuevo el corpus con el vectorizador guardado da la misma matriz (es lo que se hará con las consultas en la Fase 4).

In [11]:
def load_model(field: str, ngram: str, repr_: str = REPR) -> dict:
    with open(model_path(field, ngram, repr_), "rb") as f:
        return pickle.load(f)


for field, col in FIELDS.items():
    for ngram, (n, _) in NGRAMS.items():
        m = load_model(field, ngram)
        X = m["matrix"]

        # 1. forma y doc_ids
        assert m["doc_ids"] == doc_ids
        assert X.shape == (len(corpus), len(m["vectorizer"].vocabulary_))

        # 2. suma de la fila = número de n-gramas del documento
        n_tokens = corpus[col].str.split().str.len().to_numpy()
        expected = np.maximum(n_tokens - (n - 1), 0)
        assert (X.sum(axis=1).A1 == expected).all()

        # 3. frecuencia > 0 == binaria
        if model_path(field, ngram, "binary").exists():
            b = load_model(field, ngram, "binary")
            assert b["vectorizer"].vocabulary_ == m["vectorizer"].vocabulary_
            assert ((X > 0) != b["matrix"]).nnz == 0
        else:
            print("Aviso: falta", model_path(field, ngram, "binary").name, "(ejecutar el notebook 04)")

        # 4. normalize(frecuencia * idf) == TF-IDF
        if model_path(field, ngram, "tfidf").exists():
            t = load_model(field, ngram, "tfidf")
            assert t["vectorizer"].vocabulary_ == m["vectorizer"].vocabulary_
            rebuilt = normalize(X.multiply(t["vectorizer"].idf_).tocsr())
            assert abs(rebuilt - t["matrix"]).max() < 1e-12
        else:
            print("Aviso: falta", model_path(field, ngram, "tfidf").name, "(ejecutar el notebook 03)")

        # 5. transformar de nuevo reproduce la matriz
        X2 = m["vectorizer"].transform(corpus[col])
        assert (X != X2).nnz == 0
print("Verificaciones correctas")

Verificaciones correctas


## 8. Similitud coseno con vectores de frecuencia

La matriz se guarda **sin normalizar**, igual que la binaria. Con conteos:

$$
\cos(d, q) = \frac{\sum_j \mathrm{tf}(t_j, d)\,\mathrm{tf}(t_j, q)}{\lVert d \rVert\,\lVert q \rVert}
\qquad
\lVert d \rVert = \sqrt{\sum_j \mathrm{tf}(t_j, d)^2}
$$

- El numerador premia a los documentos que **repiten** los términos de la consulta.
- La norma crece con la repetición de cualquier término, así que un documento que repite mucho otras palabras queda penalizado.
- Todos los términos pesan igual (no hay idf): `use` cuenta lo mismo que `misinformation`.

En la Fase 4 basta con normalizar filas con `sklearn.preprocessing.normalize` antes del producto punto.

### 8.1 Comprobación de la fórmula

Se calcula el coseno con `normalize` y con la fórmula escrita a mano, y deben coincidir.

In [12]:
def cosine_scores(model: dict, query_norm: str) -> np.ndarray:
    """Coseno entre la consulta (ya normalizada) y cada documento."""
    q = model["vectorizer"].transform([query_norm])
    return (normalize(model["matrix"]) @ normalize(q).T).toarray().ravel()


m = load_model("abstract", "unigram")
query_norm = "misinformation detection use transformer"  # ya normalizada a mano
q = m["vectorizer"].transform([query_norm])
print("Términos de la consulta en el vocabulario:", q.nnz)

scores = cosine_scores(m, query_norm)

# Fórmula a mano: producto punto de conteos / (||d|| * ||q||)
dot = (m["matrix"] @ q.T).toarray().ravel()
doc_norms = np.sqrt(m["matrix"].multiply(m["matrix"]).sum(axis=1)).A1
q_norm = np.sqrt(q.multiply(q).sum())
assert np.allclose(scores, dot / (doc_norms * q_norm))
print("normalize() y la fórmula a mano coinciden")

Términos de la consulta en el vocabulario: 3
normalize() y la fórmula a mano coinciden


### 8.2 Top-10 con la representación de frecuencia

Orden descendente por puntaje y desempate estable por `arxiv_id` (decisión 4 del plan). La columna `producto_punto` es el numerador: la suma de los conteos en el documento de los términos de la consulta.

In [13]:
def ranking(scores: np.ndarray, k: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"arxiv_id": doc_ids, "score": scores})
    df = df.sort_values(["score", "arxiv_id"], ascending=[False, True], kind="stable").head(k)
    df.insert(0, "rank", range(1, k + 1))
    return df


top = ranking(scores)
top["producto_punto"] = dot[top.index]
top["norma_doc"] = doc_norms[top.index].round(2)
top["title"] = corpus.loc[top.index, "title"]
top["score"] = top["score"].round(4)
top_frequency = top.reset_index(drop=True)
top_frequency

,rank,arxiv_id,score,producto_punto,norma_doc,title
0,1,2609.39585,0.3146,7,12.85,A Generalizable and Explainable Framework for ...
1,2,2609.39662,0.2062,5,14.00,Typographic Attack Against VLM-based AI-genera...
2,3,2609.39813,0.1843,7,21.93,Backward-State Policy Is Part of the Learning ...
3,4,2609.39427,0.1826,5,15.81,PCB-MC: Missing Component Analysis in Printed ...
4,5,2610.00417,0.1722,5,16.76,Source Identification Is Not Fitness Testing: ...
5,6,2609.39385,0.1654,4,13.96,"TTLab at Daleel 2026: STAR-Ar, Sequence Taggin..."
6,7,2610.00564,0.1579,4,14.63,Attention Kernels for Learning Maps Between He...
7,8,2610.08842,0.1509,6,22.96,Beyond Risk Prediction: Evidence Grounding and...
8,9,2610.00689,0.1448,3,11.96,Towards Robust Numerical Claim Verification
9,10,2609.39142,0.1397,5,20.66,When Can Text Replace Vision? Structural Bottl...


### 8.3 Comparación de las 3 representaciones

Misma consulta y misma configuración (abstract · unigrama) con los modelos binario (notebook 04) y TF-IDF (notebook 03). La matriz TF-IDF ya tiene norma 1, así que su coseno es solo el producto punto; las otras dos se normalizan con `cosine_scores`.

In [14]:
available = {"frecuencia": top_frequency["arxiv_id"].tolist()}
if model_path("abstract", "unigram", "binary").exists():
    available["binaria"] = ranking(cosine_scores(load_model("abstract", "unigram", "binary"), query_norm))["arxiv_id"].tolist()
if model_path("abstract", "unigram", "tfidf").exists():
    t = load_model("abstract", "unigram", "tfidf")
    t_scores = (t["matrix"] @ t["vectorizer"].transform([query_norm]).T).toarray().ravel()
    available["tfidf"] = ranking(t_scores)["arxiv_id"].tolist()

comparison = pd.DataFrame({"rank": range(1, 11), **available})
for name, ids in available.items():
    if name != "frecuencia":
        common = len(set(ids) & set(available["frecuencia"]))
        print(f"En común frecuencia vs {name}: {common}/10")
comparison

En común frecuencia vs binaria: 5/10
En común frecuencia vs tfidf: 6/10


,rank,frecuencia,binaria,tfidf
0,1,2609.39585,2609.39385,2609.39585
1,2,2609.39662,2610.00564,2609.39662
2,3,2609.39813,2609.39662,2609.39427
3,4,2609.39427,2610.10556,2609.39385
4,5,2610.00417,2609.25306,2610.00564
5,6,2609.39385,2609.39585,2609.39184
6,7,2610.00564,2610.00417,2609.39429
7,8,2610.08842,2609.34802,2609.39807
8,9,2610.00689,2609.39184,2609.39627
9,10,2609.39142,2609.39635,2610.00417


## 9. Reporte de la representación de frecuencia

Se guarda en `results/frequency_stats.md`. El vocabulario de cada configuración es el mismo que en `results/tfidf_stats.md` y `results/binary_stats.md`.

In [15]:
from datetime import datetime, timezone

report = f"""# Representación de frecuencia

- Generado: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}
- scikit-learn {sklearn.__version__}, `CountVectorizer`
- Parámetros: `lowercase=False`, `token_pattern=r"\\S+"`, `binary=False`, sin `min_df`/`max_df`
- Matriz guardada sin normalizar (conteos crudos); la similitud coseno normaliza filas en la Fase 4
- Entrada: `{NORM_PATH.relative_to(ROOT)}` ({len(corpus)} documentos)
- Modelos: `models/{{campo}}_{{ngrama}}_frequency.pkl`

## Vocabulario y dispersión
{stats.to_markdown()}

- `tokens_por_doc`: n-gramas por documento (suma de la fila).
- `celdas_>1_%`: celdas no nulas con conteo > 1, donde la frecuencia difiere de la binaria.
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)

/home/mrangello/Documentos/PracticaML/results/frequency_stats.md


## 10. Resultado

Archivos generados:
- `models/title_unigram_frequency.pkl`, `models/title_bigram_frequency.pkl`, `models/abstract_unigram_frequency.pkl`, `models/abstract_bigram_frequency.pkl`: cada uno con `vectorizer`, `matrix` (CSR, conteos crudos, sin normalizar) y `doc_ids`.
- `results/frequency_stats.md`: tamaño de vocabulario, dispersión y repetición de cada matriz.

Con este notebook quedan las **12 configuraciones** de la Fase 3: {título, abstract} × {unigrama, bigrama} × {frecuencia, binaria, TF-IDF}.

**Para la Fase 4:** frecuencia y binaria se normalizan antes del producto punto (`normalize(X) @ normalize(q).T`); TF-IDF ya viene con norma L2 = 1.

**Siguiente paso:** Fase 4, recuperación con similitud coseno.